In [5]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [54]:
# Dataset Melhorado (Garante casos de fraude para a IA aprender)
import pandas as pd
import numpy as np

np.random.seed(42)
quantidade = 2000 # Dobramos os dados

dados = {
    "valor_transacao": np.random.randint(10, 10000, quantidade),
    "hora_transacao": np.random.randint(0, 24, quantidade),
    "limite_credito": np.random.randint(1000, 15000, quantidade),
    "transacoes_anteriores": np.random.randint(0, 50, quantidade),
}

df = pd.DataFrame(dados)

# Criando a regra
df["fraude"] = np.where(
    ((df["valor_transacao"] > 7000) & (df["hora_transacao"] < 5) & (df["transacoes_anteriores"] < 5)), 
    1, 0
)

# FORÇANDO MAIS CASOS DE FRAUDE (Oversampling manual)
fraudes_extras = df[df['fraude'] == 1].sample(50, replace=True)
df = pd.concat([df, fraudes_extras])

print(f"Total de fraudes no dataset: {df['fraude'].sum()}")

Total de fraudes no dataset: 57


In [55]:
df

,valor_transacao,hora_transacao,limite_credito,transacoes_anteriores,fraude
0,7280,22,1406,5,0
1,870,15,14689,35,0
2,5400,11,1166,44,0
3,5201,16,1508,35,0
4,5744,15,2661,21,0
...,...,...,...,...,...
1903,9610,0,5654,1,1
1903,9610,0,5654,1,1
1316,7574,2,5005,1,1
1768,9410,3,5471,3,1


In [56]:
X = df.drop('fraude', axis=1)
y = df['fraude']

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [58]:
from sklearn.preprocessing import StandardScaler
import joblib
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [59]:
from keras.models import Sequential
from keras.layers import Dense

In [60]:
model = Sequential()

model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

C:\Users\Karina Santos\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [62]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

# SALVE NOVAMENTE APÓS O TREINO

model.save('api/fraud_model.keras')
joblib.dump(scaler, 'api/scaler.pkl')

Epoch 1/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9596 - loss: 0.2960 - val_accuracy: 0.9756 - val_loss: 0.1024
Epoch 2/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0763 - val_accuracy: 0.9756 - val_loss: 0.0564
Epoch 3/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0486 - val_accuracy: 0.9756 - val_loss: 0.0416
Epoch 4/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0365 - val_accuracy: 0.9756 - val_loss: 0.0344
Epoch 5/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0318 - val_accuracy: 0.9756 - val_loss: 0.0315
Epoch 6/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0296 - val_accuracy: 0.9756 - val_loss: 0.0288
Epoch 7/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0279 - val_accuracy: 0.9756 - val_loss: 0.0272
Epoch 8/100
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9741 - loss: 0.0266 - val_accu

['api/scaler.pkl']

In [63]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 7.1826e-04  
Accuracy: 1.0


In [64]:
predictions = model.predict(X_test)

print(predictions)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
[[3.08636856e-38]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [9.48385726e-09]
 [1.15038138e-33]
 [6.16678883e-33]
 [7.38285834e-30]
 [1.62371257e-29]
 [5.94345188e-33]
 [5.75108613e-32]
 [0.00000000e+00]
 [1.15106561e-11]
 [0.00000000e+00]
 [1.40797431e-06]
 [6.32851834e-38]
 [3.14071590e-31]
 [2.36727205e-34]
 [0.00000000e+00]
 [4.24169800e-26]
 [0.00000000e+00]
 [1.04707149e-16]
 [3.35644435e-34]
 [0.00000000e+00]
 [1.41830794e-30]
 [0.00000000e+00]
 [2.60092044e-35]
 [1.03315557e-26]
 [0.00000000e+00]
 [9.84028280e-01]
 [1.00768540e-23]
 [1.09932245e-27]
 [7.72480909e-31]
 [9.84028280e-01]
 [8.49527282e-10]
 [9.41847232e-21]
 [1.72779651e-33]
 [9.87983724e-32]
 [3.27616514e-20]
 [0.00000000e+00]
 [1.94184089e-28]
 [2.98081622e-31]
 [0.00000000e+00]
 [1.27221922e-11]
 [6.09055973e-30]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.84941592e-34]
 [0.00000000e+00]
 [9.51208919e-02]
 [1.96654580e-35]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.70

In [65]:
model.save('fraud_model.keras')

In [66]:
from sklearn.metrics import classification_report

In [67]:
predictions = model.predict(X_test)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [68]:
predictions = (predictions > 0.5).astype(int)

In [69]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       398
           1       1.00      1.00      1.00        12

    accuracy                           1.00       410
   macro avg       1.00      1.00      1.00       410
weighted avg       1.00      1.00      1.00       410



In [70]:
nova_transacao = np.array([
 [4500, 2, 3000, 1]
])

In [71]:
nova_transacao = scaler.transform(nova_transacao)

C:\Users\Karina Santos\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [72]:
nova_transacao = pd.DataFrame([{
    'valor_transacao': 4500,
    'hora_transacao': 2,
    'limite_credito': 3000,
    'transacoes_anteriores': 1
}])

In [73]:
nova_transacao = scaler.transform(nova_transacao)

In [74]:
previsao = model.predict(nova_transacao)

print(previsao)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[3.6143183e-10]]


In [75]:
if previsao[0][0] > 0.5:
    print('🚨 Fraude detectada')
else:
    print('✅ Transação normal')

✅ Transação normal


In [76]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Separando as colunas

X = df.drop('fraude', axis=1)
y = df['fraude']

# 2. Dividindo 80% para treino e 20% para teste

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Normalizando os dados (O Scaler é essencial para redes neurais)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. SALVAR O SCALER IMEDIATAMENTE (Você precisará dele na API)

joblib.dump(scaler, 'api/scaler.pkl')
print('Scaler salvo com sucesso!')

Scaler salvo com sucesso!


In [77]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Definindo a arquitetura

model = Sequential([
    Dense(32, activation='relu', input_dim=X_train.shape[1]),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Saída entre 0 e 1 (probabilidade)
])

# Compilação

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Treinamento

print('Iniciando treinamento...')
model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2)

# SALVAR O MODELO

model.save('api/fraud_model.keras')
print('Modelo treinado e salvo na pasta api/ com sucesso!')

C:\Users\Karina Santos\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iniciando treinamento...
Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9710 - loss: 0.3670 - val_accuracy: 0.9756 - val_loss: 0.1786
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.1237 - val_accuracy: 0.9756 - val_loss: 0.0857
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9718 - loss: 0.0712 - val_accuracy: 0.9756 - val_loss: 0.0593
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9741 - loss: 0.0494 - val_accuracy: 0.9756 - val_loss: 0.0460
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9848 - loss: 0.0387 - val_accuracy: 0.9878 - val_loss: 0.0382
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9924 - loss: 0.0318 - val_accuracy: 0.9970 - val_loss: 0.0330
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9970 - loss: 0.0274 - val_accuracy: 0.9970 - val_loss: 0.0291
Epoch 8/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9977 - loss: 0.0237 -